# E-commerce Customer Behaviour Analysis and Recommendation System using PySpark
This notebook mirrors `run_project.py` step by step, so you can demo it live in a viva.
Run cells top to bottom.

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from spark_utils import get_spark
spark = get_spark('NotebookRun')
spark

## Step A+B: Ingestion + Preprocessing

In [ ]:
from preprocessing import save_cleaned_data
sales_df, cancellations_df = save_cleaned_data(spark)
sales_df.printSchema()
sales_df.show(5)

## Step C+D+E+F: Behaviour, Product, and Purchase Pattern Analysis

In [ ]:
from analysis import run_customer_behaviour_analysis, run_product_category_analysis, run_purchase_pattern_analysis
behaviour = run_customer_behaviour_analysis(spark, sales_df)
products = run_product_category_analysis(spark, sales_df)
patterns = run_purchase_pattern_analysis(spark, sales_df)
behaviour['summary']

In [ ]:
behaviour['top_customers']

In [ ]:
products['top_products_by_revenue']

## Step G: RFM + K-Means Customer Segmentation

In [ ]:
from customer_segmentation import compute_rfm, run_kmeans_segmentation, label_clusters
rfm_df = compute_rfm(spark, sales_df)
clustered, model = run_kmeans_segmentation(rfm_df, k=4)
cluster_profile, cluster_to_label = label_clusters(clustered)
cluster_profile

## Step H: ALS Recommendation System

In [ ]:
from recommendation import build_interaction_table, train_als_model, get_recommendations_for_customer
from pyspark.sql import functions as F
interactions = build_interaction_table(sales_df)
als_model, indexer_model, rmse = train_als_model(interactions)
print('RMSE:', rmse)

In [ ]:
sample_customer_id = interactions.groupBy('CustomerID').count().orderBy(F.desc('count')).first()['CustomerID']
recs = get_recommendations_for_customer(spark, als_model, indexer_model, sales_df, sample_customer_id, n=5)
recs.toPandas()

## Step I: Visualization
Run `python ../src/visualization.py` from a terminal after this notebook (it reads the CSVs saved under `output/results/`), or call the plotting functions directly below.

In [ ]:
import visualization as viz
viz.plot_top_products()
viz.plot_revenue_by_country()
viz.plot_spending_distribution()
viz.plot_purchase_trend()
viz.plot_segment_distribution()
viz.plot_recommendation_example()

In [ ]:
spark.stop()